# Strategy 3 — Collaborative Filtering

Strategy 3 walks three hops through *another user*: from the anchor user, through their 4★+ ratings, to **peer users** who rated the same movies 4★+, then back out to *other* movies those peers also rated 4★+ — movies the anchor hasn't rated. The bridge is no longer content (genres, directors, actors) — it's another person's taste.

A few notes on what this notebook is and isn't:

- **Hybrid peer definition.** Peers are users who share at least `min_overlap` ≥4★ ratings with the anchor — a *quality floor* that kills random co-occurrence noise. Of those, we keep only the top `top_n_peers` by overlap count — a *volume cap* that prevents long-tail peers from diluting the score. Both knobs become teaching levers later in the notebook.
- **Combined-product score.** `score = sum(peer.overlap × peer.rating_of_rec)` across every peer who voted for the rec. Direct mirror of Strategy 1/2's `sum(affinity)` shape — both peer strength (overlap) and peer enthusiasm (rating) are load-bearing.
- **Per-rec user-mediated chains.** Each rec carries `via_peers` (the list of contributing peers with their overlap counts) and up to 3 `peer_chains` of the form *"<shared_title> → peer <id> → <rec>"*. Same structural shape as Strategy 2's `via_people` / `seed_chains`, just routed through users instead of through people.
- **Anchor user: 196.** Same recurring anchor as notebooks 09 / 10 / 11 / 12.
- **Notebook structure: build the query phase by phase first.** Cells 4–6 run Phase 1 / Phase 2 / Phase 3 as standalone Cypher with their own `RETURN`. Cells 7–9 then reveal the unified `COLLABORATIVE_CYPHER` constant and confirm the wrapper function returns the same rows.

In [1]:
# 1. Setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from graph import run_query, close_driver

ANCHOR_USER = 196
print(f"Anchor user: {ANCHOR_USER}")

Anchor user: 196


In [2]:
# 2. User 196's peer profile — what we expect Strategy 3 to surface
top_peers = run_query(
    """
    MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
    WHERE r1.rating >= 4.0 AND r2.rating >= 4.0 AND peer <> u
    WITH peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared
    RETURN peer.userId AS peer_id, overlap, size(shared) AS n_shared_titles
    ORDER BY overlap DESC
    LIMIT 10
    """,
    {"userId": ANCHOR_USER},
)
print("Top 10 peers of user 196 by overlap (shared >=4* movies):")
print(pd.DataFrame(top_peers))

Top 10 peers of user 196 by overlap (shared >=4* movies):
   peer_id  overlap  n_shared_titles
0      303       13               13
1      416       12               12
2      758       12               12
3      308       12               12
4      276       12               12
5      151       11               11
6       94       11               11
7       43       10               10
8      342       10               10
9        7       10               10


## Let's build the query phase by phase

Before showing the unified query (cell 8) and calling the wrapper function (cell 9), let's run each phase as its own Cypher block:

- **Phase 1 — peer pool.** From the anchor's ≥4★ ratings, find peers who also rated those same movies ≥4★. Group by peer, count distinct shared movies, drop anyone below `min_overlap`, keep the top `top_n_peers` by overlap. Output: a list of `{peer_id, overlap, shared}` rows.
- **Phase 2 — row-level rec evidence.** For each top peer, find their ≥4★ ratings on movies the anchor hasn't rated. Output: one row per `(rec, peer)` vote, with the peer's overlap and their rating of that rec. This is the row-level evidence Phase 3 will aggregate.
- **Phase 3 — score and rank.** Group the rows from Phase 2 by rec, compute `sum(overlap × peer_rating)`, build `via_peers` + `peer_chains`, return ranked.

Each cell below ends with `RETURN` so it runs standalone. In the unified version (cell 8), those `RETURN`s become `WITH` — same callback to the `WITH` ↔ `RETURN` distinction from notebook 09's Cypher 101 walkthrough.

In [3]:
# 4. Phase 1 standalone — peer pool: floor + cap on overlap
PHASE_1 = """
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
WHERE r1.rating >= $threshold
  AND r2.rating >= $threshold
  AND peer <> u
WITH u, peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared_titles
WHERE overlap >= $min_overlap
WITH u, peer, overlap, shared_titles
ORDER BY overlap DESC
LIMIT $top_n_peers
RETURN peer.userId AS peer_id, overlap, size(shared_titles) AS n_shared_titles
"""

phase_1_rows = run_query(
    PHASE_1,
    {"userId": ANCHOR_USER, "threshold": 4.0, "min_overlap": 3, "top_n_peers": 25},
)
print(f"Phase 1 — {len(phase_1_rows)} peers passed the floor (min_overlap>=3) and cap (top 25):")
print(pd.DataFrame(phase_1_rows).head(10))

Phase 1 — 25 peers passed the floor (min_overlap>=3) and cap (top 25):
   peer_id  overlap  n_shared_titles
0      303       13               13
1      416       12               12
2      308       12               12
3      276       12               12
4      758       12               12
5       94       11               11
6      151       11               11
7      591       10               10
8      711       10               10
9      474       10               10


In [4]:
# 5. Phase 2 standalone — row-level rec evidence: one row per (peer, rec) vote
PHASE_2 = """
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
WHERE r1.rating >= $threshold
  AND r2.rating >= $threshold
  AND peer <> u
WITH u, peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared_titles
WHERE overlap >= $min_overlap
WITH u, peer, overlap, shared_titles
ORDER BY overlap DESC
LIMIT $top_n_peers
WITH u, collect({peer_id: peer.userId, overlap: overlap, shared: shared_titles}) AS top_peers

UNWIND top_peers AS tp
MATCH (peer:User {userId: tp.peer_id})-[r3:RATED]->(rec:Movie)
WHERE r3.rating >= $threshold
  AND NOT EXISTS { (u)-[:RATED]->(rec) }
RETURN rec.title AS rec_title,
       tp.peer_id AS peer_id,
       r3.rating  AS peer_rating,
       tp.overlap AS overlap
ORDER BY overlap DESC, peer_rating DESC, rec_title
"""

phase_2_rows = run_query(
    PHASE_2,
    {"userId": ANCHOR_USER, "threshold": 4.0, "min_overlap": 3, "top_n_peers": 25},
)
print(f"Phase 2 — {len(phase_2_rows)} (peer, rec) votes before grouping. First 15 rows:")
print(pd.DataFrame(phase_2_rows).head(15))

Phase 2 — 4552 (peer, rec) votes before grouping. First 15 rows:
                rec_title  peer_id  peer_rating  overlap
0   Age of Innocence, The      303          5.0       13
1                 Aladdin      303          5.0       13
2                   Alien      303          5.0       13
3                  Aliens      303          5.0       13
4                 Amadeus      303          5.0       13
5              Annie Hall      303          5.0       13
6        Army of Darkness      303          5.0       13
7                  Batman      303          5.0       13
8    Beauty and the Beast      303          5.0       13
9          Big Sleep, The      303          5.0       13
10          Billy Madison      303          5.0       13
11                  Bound      303          5.0       13
12             Braveheart      303          5.0       13
13     Breaking the Waves      303          5.0       13
14  Bullets Over Broadway      303          5.0       13


In [5]:
# 6. Phase 3 standalone — group by rec, score with combined product, build chains
PHASE_3 = """
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
WHERE r1.rating >= $threshold
  AND r2.rating >= $threshold
  AND peer <> u
WITH u, peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared_titles
WHERE overlap >= $min_overlap
WITH u, peer, overlap, shared_titles
ORDER BY overlap DESC
LIMIT $top_n_peers
WITH u, collect({peer_id: peer.userId, overlap: overlap, shared: shared_titles}) AS top_peers

UNWIND top_peers AS tp
MATCH (peer:User {userId: tp.peer_id})-[r3:RATED]->(rec:Movie)
WHERE r3.rating >= $threshold
  AND NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, tp, r3.rating AS peer_rating

WITH rec, collect({
  peer_id: tp.peer_id,
  overlap: tp.overlap,
  peer_rating: peer_rating,
  shared: tp.shared
}) AS contributing
RETURN rec.title AS title,
       rec.year  AS year,
       reduce(s = 0.0, c IN contributing | s + (c.overlap * c.peer_rating)) AS score,
       [c IN contributing | {peer_id: c.peer_id, overlap: c.overlap}] AS via_peers,
       [c IN contributing | head(c.shared) + ' → peer ' + toString(c.peer_id) + ' → ' + rec.title][..3] AS peer_chains
ORDER BY score DESC, title ASC
LIMIT $k
"""

phase_3_rows = run_query(
    PHASE_3,
    {"userId": ANCHOR_USER, "threshold": 4.0, "min_overlap": 3, "top_n_peers": 25, "k": 10},
)

def render(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "year", "score", "via_peers", "peer_chains"])
    df = pd.DataFrame(rows)
    df["via_peers"] = df["via_peers"].apply(
        lambda xs: ", ".join(f"{p['peer_id']}({p['overlap']})" for p in xs)
    )
    df["peer_chains"] = df["peer_chains"].apply(lambda xs: "\n".join(xs))
    return df[["title", "year", "score", "via_peers", "peer_chains"]]

render(phase_3_rows)

,title,year,score,via_peers,peer_chains
0,Star Wars,1977.0,1080.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Star Wars\nBabe → peer 416 →...
1,Back to the Future,1985.0,1048.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Back to the Future\nBabe → p...
2,Raiders of the Lost Ark,1981.0,1028.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Raiders of the Lost Ark\nBab...
3,Amadeus,1984.0,1020.0,"303(13), 416(12), 308(12), 758(12), 94(11), 59...",Babe → peer 303 → Amadeus\nBabe → peer 416 → A...
4,Fargo,1996.0,1000.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Fargo\nBabe → peer 416 → Far...
5,"Shawshank Redemption, The",1994.0,994.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...","Babe → peer 303 → Shawshank Redemption, The\nB..."
6,"Empire Strikes Back, The",1980.0,983.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...","Babe → peer 303 → Empire Strikes Back, The\nBa..."
7,Return of the Jedi,1983.0,974.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Return of the Jedi\nBabe → p...
8,Monty Python and the Holy Grail,1974.0,911.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Monty Python and the Holy Gr...
9,Pulp Fiction,1994.0,905.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Pulp Fiction\nBabe → peer 41...


## Now bring it all together

Phases 1–3 above are the body of the unified `COLLABORATIVE_CYPHER` constant in [`src/recommender.py`](../src/recommender.py). The collapse is mechanical:

- Each standalone phase ended with `RETURN` so it could run on its own.
- In the unified query, those `RETURN`s become `WITH` — the same variables flow into the next phase instead of leaving the query.
- The final `RETURN` lives at the end of Phase 3, producing the rec list.

Cell 8 prints the constant. Cell 9 calls the wrapper function and confirms the output matches Phase 3 above row-for-row.

In [6]:
# 8. The unified query — same Cypher you just ran phase by phase, chained with WITH
from recommender import COLLABORATIVE_CYPHER
print(COLLABORATIVE_CYPHER)


// Phase 1 — find peers: users sharing >= min_overlap >=-threshold-rated movies
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
WHERE r1.rating >= $threshold
  AND r2.rating >= $threshold
  AND peer <> u
WITH u, peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared_titles
WHERE overlap >= $min_overlap
WITH u, peer, overlap, shared_titles
ORDER BY overlap DESC
LIMIT $top_n_peers
WITH u, collect({
  peer_id: peer.userId,
  overlap: overlap,
  shared: shared_titles
}) AS top_peers

// Phase 2 — for each top peer, find their >=-threshold-rated movies the user hasn't seen
UNWIND top_peers AS tp
MATCH (peer:User {userId: tp.peer_id})-[r3:RATED]->(rec:Movie)
WHERE r3.rating >= $threshold
  AND NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, tp, r3.rating AS peer_rating

// Phase 3 — group by rec, score with combined product, build explainability
WITH rec, collect({
  peer_id: tp.peer_id,
  overlap: tp.overlap,
  peer_rating: peer_rating,

In [7]:
# 9. Call the wrapper — should match Phase 3 cell 6 row-for-row
from recommender import collaborative_filtering
recs = collaborative_filtering(ANCHOR_USER)
render(recs)

,title,year,score,via_peers,peer_chains
0,Star Wars,1977.0,1080.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Star Wars\nBabe → peer 416 →...
1,Back to the Future,1985.0,1048.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Back to the Future\nBabe → p...
2,Raiders of the Lost Ark,1981.0,1028.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Raiders of the Lost Ark\nBab...
3,Amadeus,1984.0,1020.0,"303(13), 416(12), 308(12), 758(12), 94(11), 59...",Babe → peer 303 → Amadeus\nBabe → peer 416 → A...
4,Fargo,1996.0,1000.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Fargo\nBabe → peer 416 → Far...
5,"Shawshank Redemption, The",1994.0,994.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...","Babe → peer 303 → Shawshank Redemption, The\nB..."
6,"Empire Strikes Back, The",1980.0,983.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...","Babe → peer 303 → Empire Strikes Back, The\nBa..."
7,Return of the Jedi,1983.0,974.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Return of the Jedi\nBabe → p...
8,Monty Python and the Holy Grail,1974.0,911.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Monty Python and the Holy Gr...
9,Pulp Fiction,1994.0,905.0,"303(13), 416(12), 308(12), 276(12), 758(12), 9...",Babe → peer 303 → Pulp Fiction\nBabe → peer 41...


## Reading one explainability chain

Pick the top rec from cell 9. Read its `via_peers` first — a list of `{peer_id, overlap}` records telling you *who* voted for this rec and *how strongly each voter overlaps with the anchor's taste*. A rec with 5 contributing peers, each overlapping on 8–12 movies, is a much stronger endorsement than a rec with 1 peer overlapping on 3 movies — and the `score` (`sum(overlap × peer_rating)`) reflects that.

Then narrate one `peer_chain` end-to-end:

> *"User 196 rated **<shared_title>** 4★+ — and so did **peer <id>** (who overlaps with user 196 on **<overlap>** other 4★+ movies) — and **peer <id>** also rated **<rec>** 4★+, which user 196 hasn't seen → here's the rec."*

The contrast with Strategy 2 is the point of this notebook. Strategy 2's chain answers *"because of this director/actor"*; Strategy 3's chain answers *"because people like you loved it."* Same query shape, different bridging entity, different question. Strategy 4 will weight them together.

Two more things to internalize:

1. **`score` is a ranking signal, not a confidence percentage.** A score of 240 means *"sum of (overlap × peer_rating) across contributing peers = 240"* — useful for ordering, not for probability claims. Strategy 4 will need to normalize before combining signals.
2. **`peer_chains` is capped at 3 in unwind order**, not strength-ranked. Peers are unwound in `overlap DESC` order, so position-1 chain comes from the highest-overlap peer, not necessarily the one whose rating contributed the most score. All three chains are real; none is *"the strongest contribution."*

In [9]:
# 11. Knob — min_overlap (the quality floor on the peer pool)
loose = collaborative_filtering(ANCHOR_USER, min_overlap=1)
tight = collaborative_filtering(ANCHOR_USER, min_overlap=10)

print("min_overlap=1 — random-coincidence peers flood in; top recs become popularity-driven:")
print(render(loose).head(3))
print(f"\n(total recs at min_overlap=1: {len(loose)})")
print("\nmin_overlap=10 — strict floor; only deeply-shared-taste peers contribute:")
print(render(tight).head(3))
print(f"\n(total recs at min_overlap=10: {len(tight)})")
print("\nThe default min_overlap=3 sits between these extremes — the floor kills noise,")
print("but doesn't strangle the peer pool for a typical user.")

min_overlap=1 — random-coincidence peers flood in; top recs become popularity-driven:
                       title    year   score  \
0         Back to the Future  1985.0  1093.0   
1                  Star Wars  1977.0  1089.0   
2  Shawshank Redemption, The  1994.0  1039.0   

                                           via_peers  \
0  303(13), 758(12), 276(12), 416(12), 308(12), 1...   
1  303(13), 758(12), 276(12), 416(12), 308(12), 1...   
2  303(13), 758(12), 276(12), 416(12), 308(12), 1...   

                                         peer_chains  
0  Babe → peer 303 → Back to the Future\nBabe → p...  
1  Babe → peer 303 → Star Wars\nBabe → peer 758 →...  
2  Babe → peer 303 → Shawshank Redemption, The\nB...  

(total recs at min_overlap=1: 10)

min_overlap=10 — strict floor; only deeply-shared-taste peers contribute:
                title    year  score  \
0  Back to the Future  1985.0  796.0   
1           Star Wars  1977.0  783.0   
2             Amadeus  1984.0  768.0   

     

In [10]:
# 12. Knob — top_n_peers (the volume cap on the peer pool)
narrow = collaborative_filtering(ANCHOR_USER, top_n_peers=5)
wide = collaborative_filtering(ANCHOR_USER, top_n_peers=100)

print("top_n_peers=5 — only the 5 highest-overlap peers contribute; via_peers lists are short:")
print(render(narrow).head(3))
print("\ntop_n_peers=100 — tail peers (lower overlap, weaker bridges) now contribute to the score:")
print(render(wide).head(3))
print("\nThe cap interacts with min_overlap: a high floor makes the cap less binding")
print("because fewer peers survive the floor in the first place. Default top_n_peers=25")
print("is wide enough for diverse recs without long-tail dilution.")

top_n_peers=5 — only the 5 highest-overlap peers contribute; via_peers lists are short:
          title    year  score                                    via_peers  \
0         Fargo  1996.0  305.0  303(13), 416(12), 276(12), 308(12), 758(12)   
1  Pulp Fiction  1994.0  305.0  303(13), 416(12), 276(12), 308(12), 758(12)   
2   Rear Window  1954.0  305.0  303(13), 416(12), 276(12), 308(12), 758(12)   

                                         peer_chains  
0  Babe → peer 303 → Fargo\nBabe → peer 416 → Far...  
1  Babe → peer 303 → Pulp Fiction\nBabe → peer 41...  
2  Babe → peer 303 → Rear Window\nBabe → peer 416...  

top_n_peers=100 — tail peers (lower overlap, weaker bridges) now contribute to the score:
                     title    year   score  \
0                Star Wars  1977.0  3113.0   
1                    Fargo  1996.0  2999.0   
2  Raiders of the Lost Ark  1981.0  2833.0   

                                           via_peers  \
0  303(13), 308(12), 416(12), 276(12), 758(

In [11]:
# 13. Edge case — cold start
# 13a. A user with zero >=4* ratings, if one exists in the dataset
no_signal_user = run_query(
    """
    MATCH (u:User)
    WHERE NOT EXISTS { MATCH (u)-[r:RATED]->() WHERE r.rating >= 4 }
    RETURN u.userId AS userId LIMIT 1
    """,
)
if no_signal_user:
    cold_id = no_signal_user[0]["userId"]
    n = len(collaborative_filtering(cold_id))
    print(f"13a. User {cold_id} has zero >=4* ratings -> collaborative_filtering returned {n} recs.")
else:
    print("13a. (No >=4*-free user in MovieLens 100K — skipping.)")

# 13b. Non-existent user
n_ghost = len(collaborative_filtering(99999))
print(f"13b. User 99999 doesn't exist -> collaborative_filtering returned {n_ghost} recs.")

print("\nContract: empty list = no signal. Same as Strategies 1 and 2; Strategy 4 owns the fallback.")

13a. User 685 has zero >=4* ratings -> collaborative_filtering returned 0 recs.
13b. User 99999 doesn't exist -> collaborative_filtering returned 0 recs.

Contract: empty list = no signal. Same as Strategies 1 and 2; Strategy 4 owns the fallback.


In [12]:
# 14. Anchor sanity — none of the recs should already be rated by user 196
rec_ids = [r["movieId"] for r in recs]
overlap = run_query(
    """
    MATCH (u:User {userId: $userId})-[:RATED]->(m:Movie)
    WHERE m.movieId IN $ids
    RETURN count(m) AS overlap
    """,
    {"userId": ANCHOR_USER, "ids": rec_ids},
)
n = overlap[0]["overlap"] if overlap else -1
print("OK — 0 overlap" if n == 0 else f"MISMATCH — user 196 has already rated {n} of these recs")

OK — 0 overlap


In [13]:
# 15. Cleanup
close_driver()
print("Driver closed.")

Driver closed.
